In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
  confusion_matrix,classification_report,
  f1_score,roc_auc_score
)
from pathlib import Path
import warnings
#suppress warnings
warnings.filterwarnings('ignore')
print("Imports successful")

Imports successful


In [11]:
dataset = pd.read_csv(Path("../classification/Social_Network_Ads.csv"))
dataset.head()

,User ID,Gender,Age,EstimatedSalary,Purchased
0,15624510,Male,19,19000,0
1,15810944,Male,35,20000,0
2,15668575,Female,26,43000,0
3,15603246,Female,27,57000,0
4,15804002,Male,19,76000,0


In [12]:
dataset=pd.get_dummies(dataset,drop_first=True)
dataset.head()

,User ID,Age,EstimatedSalary,Purchased,Gender_Male
0,15624510,19,19000,0,True
1,15810944,35,20000,0,True
2,15668575,26,43000,0,False
3,15603246,27,57000,0,False
4,15804002,19,76000,0,True


In [13]:
dataset=dataset.drop("User ID",axis=1)
dataset.head()
print(f"dataset.shape: {dataset.shape}")

dataset.shape: (400, 4)


Why are we passing a 2-d array for indep and only 1-d array for dependent?

In machine learning classification, independent variables (features) form a 2-D array/matrix because each row represents a sample (e.g., a user) and each column represents a feature (e.g., Age, EstimatedSalary, Gender_Male)

Dependent variables (target, like Purchased) stay 1-D because they are scalar labels per sample(0 or 1 here)


In [ ]:
indep=dataset[["Age","EstimatedSalary","Gender_Male"]]
dep=dataset["Purchased"]
print(f"indep.shape: {indep.shape} ")
print(f"dep.shape: {dep.shape} ")

In [ ]:
#split into training set and test
X_train, X_test, y_train, y_test = train_test_split(indep, dep, test_size = 1/3, random_state = 0)

In [ ]:
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [ ]:
param_grid = {'solver':['newton-cg', 'lbfgs', 'liblinear', 'saga'],
             'penalty':['l2']} 

grid = GridSearchCV(LogisticRegression(), param_grid, refit = True, verbose = 3,n_jobs=-1,scoring='f1_weighted') 

# fitting the model for grid search 
grid.fit(X_train, y_train) 


In [ ]:
#print best parameter
print(f"Best parameter={grid.best_params_}")

In [ ]:
#Let us see the results in tabular format
res_dict = grid.cv_results_
res_table = pd.DataFrame.from_dict(res_dict)

# Display only first 5 rows
print("First 5 rows:")
first_five_rows=res_table.head(5)
first_five_rows

In [ ]:
# Display the row that has rank_test_score = 1
# verify this with the best parameters that we displayed
# The parameter column names will be prefixed with "param_"
# Same info is also displayed under "params" column 

print("\nRow with rank_test_score = 1:")
res_table[res_table['rank_test_score'] == 1]


In [ ]:
#Run prediction
y_pred = grid.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
clf_report = classification_report(y_test, y_pred)

In [ ]:
f1_macro=f1_score(y_test,y_pred,average='weighted')
print(f"f1 macro = {f1_macro}")

In [ ]:
print("Confusion Matix:\n", cm)
print("Classification Report:\n",clf_report)


In [ ]:
roc_score = roc_auc_score(y_test,grid.predict_proba(X_test)[:,1])
print(f"roc_auc score:{roc_score}")

In [ ]:
age_input=47
estimated_salary_input=49000
sex_male_input=False


In [ ]:
transformed_input = sc.transform([[age_input,estimated_salary_input,sex_male_input]])
predicted = grid.predict(transformed_input)
print("predicted=",predicted)